# Separate final-test evaluation
Default: CPU-only existence preflight, no test labels or metrics. Stop after the first cell until final evaluation is separately approved. The second cell defaults to denied approval and CPU inference. It uses frozen validation-selected weights, identical 200-way cosine pools and FP32 for all arms. No training, extraction, text supervision or vision model download. Results are new; interrupted evaluation directories are preserved and require a new OUT name. Completed identical runs return saved results.


In [ ]:
CODE_REVISION = '9b3accad2bcf3a76a77936aaea30e91469acfd48'
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO=Path(tempfile.mkdtemp(prefix='audit-eval-',dir='/content'))/'repo'
subprocess.run(['git','clone','--no-checkout','https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',CODE_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_REVISION
WORK=Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP=WORK/'pilot_sub01_200concepts_v1'
TRAINING=WORK/'controlled_sub01_five_epochs_v1'
# No catalog/provenance deserialization or test-label access before approval.
subprocess.run([sys.executable,'-m','phase1.final_audit','--training',str(TRAINING),
 '--dataset',str(PREP/'dataset.json')],cwd=REPO,env=dict(os.environ,CUDA_VISIBLE_DEVICES=''),check=True)


In [ ]:
APPROVE_FINAL_TEST_EVALUATION = False #@param {type:"boolean"}
assert APPROVE_FINAL_TEST_EVALUATION, 'Stop: explicit approval required before test-label access or metrics.'
import time, torch
DEVICE = 'cpu' #@param ['cpu','cuda']
CU_PER_HOUR = 0.0 #@param {type:"number"}
CU_CAP = 10.0 #@param {type:"number"}
TIME_CAP_HOURS = 2.0 #@param {type:"number"}
if DEVICE=='cuda':
    assert torch.cuda.is_available()
    assert CU_PER_HOUR>0 and CU_CAP>0, 'Enter actual resource rate and approved cap.'
    assert torch.cuda.mem_get_info()[0]>4*2**30, 'Need free GPU memory; inference uses batch8/chunk1.'
assert TIME_CAP_HOURS>0
OUT=WORK/'final_test_five_epoch_v1'
LOGS=WORK/'final_test_five_epoch_v1_logs'
LEDGER=LOGS/'resource_usage.jsonl'
prior=max([json.loads(line).get('estimated_cu') or 0 for line in LEDGER.read_text().splitlines() if line.strip()],default=0.) if LEDGER.exists() else 0.
os.environ.update(PHASE1_SESSION_START=str(time.time()),PHASE1_CU_RATE=str(CU_PER_HOUR),
 PHASE1_CU_LIMIT=str(CU_CAP),PHASE1_CU_PRIOR=str(prior),PHASE1_SECONDS_LIMIT=str(TIME_CAP_HOURS*3600),PHASE1_RESOURCE_LOG=str(LEDGER))
sys.path.insert(0,str(REPO))
from phase1.process_log import run_logged
from phase1.patch_resources import BudgetMeter
meter=BudgetMeter()
run_logged([sys.executable,'-m','phase1.final_audit','--training',str(TRAINING),
 '--dataset',str(PREP/'dataset.json'),'--output',str(OUT),'--approve-final-test','--device',DEVICE],
 REPO,LOGS,check_budget=lambda:meter.check('final_test',force=True))
print('Final-test result location:',OUT)
